## Setup rclone untuk Upload ke Google Drive (Cell 0)

> Lakukan langkah ini **sebelum** training jika ingin upload model otomatis ke Google Drive setelah selesai.  
> Jika tidak perlu upload, lewati bagian ini dan langsung jalankan Cell 1.

**Langkah-langkah setup rclone di RunPod:**

**1. Buka Terminal baru di Jupyter** (menu: File → New → Terminal)

**2. Install rclone:**
```bash
curl https://rclone.org/install.sh | sudo bash
```

**3. Jalankan konfigurasi interaktif:**
```bash
rclone config
```

**4. Ikuti panduan konfigurasi:**
- Ketik `n` → new remote
- Masukkan nama remote, contoh: `gdrive`
- Pilih storage type: ketik `drive` (Google Drive)
- Client ID & Secret: tekan Enter (kosongkan, pakai default)
- Scope: pilih `1` (full access)
- Autentikasi: salin URL ke browser, login Google, masukkan kode verifikasi
- Konfirmasi dengan `y`

**5. Verifikasi setup berhasil:**
```bash
rclone lsd gdrive:
```

**6. Setelah setup selesai, jalankan Cell 7** untuk upload model ke Google Drive.

---
> **Catatan:** Nama remote default di Cell 7 adalah `gdrive`. Jika kamu menggunakan nama lain saat setup, ubah variabel `RCLONE_REMOTE` di Cell 7.

# Fine-Tuning TrOCR — IDP Lintasarta
**Model base:** `microsoft/trocr-base-handwritten`  
**Dataset:** 34.006 sampel (3.957 original + 30.049 augmentasi)  
**Target:** RunPod A100 80GB

---
**Urutan jalankan:**
1. Ubah `PATH_DATASET` di Cell 1 sesuai lokasi di RunPod
2. Jalankan semua cell dari atas ke bawah
3. Training otomatis resume jika ada checkpoint

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 1 — KONFIGURASI PATH & INSTALL DEPENDENCIES
# ════════════════════════════════════════════════════════════════

# ── UBAH SESUAI LOKASI DI RUNPOD ────────────────────────────────
# Contoh RunPod : '/workspace/python-engine/Dataset'
# Contoh lokal  : '/home/user/idp-lintasarta/python-engine/Dataset'
PATH_DATASET = '/workspace/python-engine/Dataset'   # <── UBAH INI

PATH_LABELS  = PATH_DATASET + '/labels_augmented.csv'
PATH_OUTPUT  = 'model_finetuned/'

import os
os.makedirs(PATH_OUTPUT, exist_ok=True)

print('PATH_DATASET :', PATH_DATASET)
print('PATH_LABELS  :', PATH_LABELS)
print('PATH_OUTPUT  :', PATH_OUTPUT)
print()

# Cek labels file ada
assert os.path.isfile(PATH_LABELS), f'File tidak ditemukan: {PATH_LABELS}'
print('labels_augmented.csv ditemukan.')

# ── INSTALL DEPENDENCIES ─────────────────────────────────────────
# Pin versi eksplisit untuk menghindari konflik di RunPod
import subprocess
subprocess.run(
    ['pip', 'install', '-q',
     'transformers==4.37.2', 'accelerate==0.25.0',
     'datasets', 'torch', 'pillow', 'jiwer', 'matplotlib', 'sentencepiece'],
    check=True
)
print('Dependencies installed.')

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 2 — IMPORT LIBRARY & SET SEED
# ════════════════════════════════════════════════════════════════

import os, csv, json, time, random
import numpy as np
import torch
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset
from transformers import (
    TrOCRProcessor,
    VisionEncoderDecoderModel,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    EarlyStoppingCallback,
    TrainerCallback,
)
from jiwer import cer as jiwer_cer
import matplotlib
matplotlib.use('Agg')   # non-interactive: aman di server
import matplotlib.pyplot as plt
from datetime import datetime

# ── SET RANDOM SEED UNTUK REPRODUCIBILITY ───────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM   : {vram:.1f} GB')
print(f'PyTorch: {torch.__version__}')

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 3 — LOAD DATASET & PERSIAPAN UNTUK MODEL
# ════════════════════════════════════════════════════════════════

# ── BACA labels_augmented.csv ────────────────────────────────────
print('Membaca dataset...')
data_all        = []
skipped_missing = []

with open(PATH_LABELS, newline='', encoding='utf-8') as f:
    for row in csv.reader(f):
        if len(row) < 2 or not row[0].strip():
            continue
        img_rel  = row[0].strip()
        label    = row[1].strip()
        img_path = os.path.join(PATH_DATASET, img_rel)
        if not os.path.isfile(img_path):
            skipped_missing.append(img_rel)
            continue
        data_all.append((img_path, label))

print(f'Total data valid   : {len(data_all)}')
print(f'File tidak ada     : {len(skipped_missing)}')
if skipped_missing:
    for p in skipped_missing[:5]:
        print(f'  SKIP: {p}')

# ── SHUFFLE & SPLIT 80% TRAIN / 20% VAL ─────────────────────────
random.seed(SEED)
random.shuffle(data_all)

split_idx  = int(0.8 * len(data_all))
data_train = data_all[:split_idx]
data_val   = data_all[split_idx:]

print(f'\nTrain : {len(data_train)} sampel  ({len(data_train)/len(data_all)*100:.1f}%)')
print(f'Val   : {len(data_val)}  sampel  ({len(data_val)/len(data_all)*100:.1f}%)')

# ── DATASET CLASS ────────────────────────────────────────────────
MAX_TARGET_LENGTH = 128   # max panjang teks (karakter)

class TrOCRDataset(Dataset):
    """
    Membaca gambar dari disk dan memproses dengan TrOCRProcessor.
    Tidak ada augmentasi tambahan — gambar dibaca apa adanya.
    """
    def __init__(self, data, processor):
        self.data      = data
        self.processor = processor
        self.n_corrupt = 0

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_path, label = self.data[idx]

        # Load gambar — fallback blank putih jika corrupt
        try:
            image = Image.open(img_path).convert('RGB')
        except Exception:
            self.n_corrupt += 1
            image = Image.new('RGB', (384, 384), color=255)

        # Proses gambar: resize + normalize sesuai requirement TrOCR
        pixel_values = self.processor(
            images=image, return_tensors='pt'
        ).pixel_values.squeeze(0)

        # Tokenisasi label teks
        labels = self.processor.tokenizer(
            label,
            padding='max_length',
            max_length=MAX_TARGET_LENGTH,
            truncation=True,
            return_tensors='pt',
        ).input_ids.squeeze(0)

        # Token padding diabaikan dalam loss (nilai -100)
        labels[labels == self.processor.tokenizer.pad_token_id] = -100

        return {'pixel_values': pixel_values, 'labels': labels}

print('\nTrOCRDataset class siap.')

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 4 — LOAD MODEL & KONFIGURASI TRAINING
# ════════════════════════════════════════════════════════════════

# ── HYPERPARAMETER ───────────────────────────────────────────────
MODEL_NAME    = 'microsoft/trocr-base-handwritten'
BATCH_SIZE    = 16       # dikurangi otomatis ke 8 jika OOM
LEARNING_RATE = 5e-5
MAX_EPOCHS    = 20
WARMUP_STEPS  = 500
WEIGHT_DECAY  = 0.01
PATIENCE      = 3        # early stopping: stop jika val_loss tidak turun N epoch
FP16          = True     # mixed precision — lebih cepat di A100

print('=== Konfigurasi Training ===')
print(f'Model         : {MODEL_NAME}')
print(f'Batch size    : {BATCH_SIZE}')
print(f'Learning rate : {LEARNING_RATE}')
print(f'Max epochs    : {MAX_EPOCHS}')
print(f'Warmup steps  : {WARMUP_STEPS}')
print(f'Weight decay  : {WEIGHT_DECAY}')
print(f'Early stop    : patience={PATIENCE} epoch')
print(f'FP16          : {FP16}')
print()

# ── LOAD PROCESSOR ───────────────────────────────────────────────
print('Loading TrOCRProcessor...')
processor = TrOCRProcessor.from_pretrained(MODEL_NAME)
print('Processor loaded.')

# ── LOAD MODEL ───────────────────────────────────────────────────
print('Loading VisionEncoderDecoderModel...')
model = VisionEncoderDecoderModel.from_pretrained(MODEL_NAME)

# Token khusus untuk decoder (wajib di-set untuk TrOCR)
model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.config.pad_token_id           = processor.tokenizer.pad_token_id
model.config.eos_token_id           = processor.tokenizer.sep_token_id
model.config.vocab_size             = model.config.decoder.vocab_size

# Konfigurasi generation (digunakan saat evaluasi dengan beam search)
model.config.max_new_tokens       = MAX_TARGET_LENGTH
model.config.early_stopping       = True
model.config.no_repeat_ngram_size = 3
model.config.length_penalty       = 2.0
model.config.num_beams            = 4

model = model.to(DEVICE)
n_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f'Model loaded: {n_params:.1f}M parameter')

# ── BUAT DATASET OBJECTS ─────────────────────────────────────────
train_dataset = TrOCRDataset(data_train, processor)
val_dataset   = TrOCRDataset(data_val,   processor)
print(f'\nTrain dataset : {len(train_dataset)} sampel')
print(f'Val dataset   : {len(val_dataset)} sampel')

# ── COMPUTE METRICS: CER ─────────────────────────────────────────
def compute_metrics(pred):
    """Hitung CER (Character Error Rate) di validation set."""
    label_ids = pred.label_ids
    pred_ids  = pred.predictions

    # Clip token IDs untuk mencegah OverflowError saat batch_decode
    vocab_size = processor.tokenizer.vocab_size
    pred_ids   = np.clip(pred_ids, 0, vocab_size - 1)

    # Decode prediksi token ke string
    pred_str = processor.batch_decode(pred_ids, skip_special_tokens=True)

    # Kembalikan -100 ke pad_token sebelum decode
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    label_str = processor.batch_decode(label_ids, skip_special_tokens=True)

    # Hindari string kosong yang bisa crash jiwer
    pred_str  = [p if p.strip() else ' ' for p in pred_str]
    label_str = [l if l.strip() else ' ' for l in label_str]

    cer_score = jiwer_cer(label_str, pred_str)
    return {'cer': round(cer_score, 4)}

print('\nSemua konfigurasi siap.')

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 5 — TRAINING LOOP
# ════════════════════════════════════════════════════════════════

# ── CALLBACK: TAMPILKAN RINGKASAN PER EPOCH ──────────────────────
class EpochSummaryCallback(TrainerCallback):
    """Tampilkan train loss, val loss, CER, dan ETA setiap akhir epoch."""

    def __init__(self):
        self.t_epoch_start = None
        self.last_train_loss = 0.0

    def on_epoch_begin(self, args, state, control, **kwargs):
        self.t_epoch_start = time.time()

    def on_log(self, args, state, control, logs=None, **kwargs):
        # Simpan train loss terakhir sebelum evaluation
        if logs and 'loss' in logs:
            self.last_train_loss = logs['loss']

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        if not metrics:
            return
        epoch     = state.epoch
        elapsed   = time.time() - (self.t_epoch_start or time.time())
        remaining = (args.num_train_epochs - epoch) * elapsed

        val_loss  = metrics.get('eval_loss', 0)
        cer_score = metrics.get('eval_cer',  0)

        print(
            f'  [Epoch {int(epoch):>2}/{int(args.num_train_epochs)}] '
            f'train_loss={self.last_train_loss:.4f}  '
            f'val_loss={val_loss:.4f}  '
            f'CER={cer_score*100:.2f}%  '
            f'ETA~{remaining/60:.0f}mnt'
        )


# ── CEK RESUME DARI CHECKPOINT ───────────────────────────────────
resume_checkpoint = None
if os.path.isdir(PATH_OUTPUT):
    ckpts = sorted(
        [d for d in os.listdir(PATH_OUTPUT) if d.startswith('checkpoint-')],
        key=lambda x: int(x.split('-')[-1])
    )
    if ckpts:
        resume_checkpoint = os.path.join(PATH_OUTPUT, ckpts[-1])
        print(f'Resume dari checkpoint: {resume_checkpoint}')
    else:
        print('Tidak ada checkpoint — mulai dari awal.')

# ── TRAINING DENGAN OOM HANDLING ─────────────────────────────────
start_time = time.time()   # digunakan oleh Cell 6 untuk menghitung total_train_time
batch_size = BATCH_SIZE
epoch_cb   = EpochSummaryCallback()

print(f'\nMulai training...')
print('=' * 60)

for attempt in range(2):   # attempt 0: batch=16, attempt 1: batch=8
    try:
        training_args = Seq2SeqTrainingArguments(
            output_dir                  = PATH_OUTPUT,
            num_train_epochs            = MAX_EPOCHS,
            per_device_train_batch_size = batch_size,
            per_device_eval_batch_size  = batch_size,
            learning_rate               = LEARNING_RATE,
            weight_decay                = WEIGHT_DECAY,
            warmup_steps                = WARMUP_STEPS,
            lr_scheduler_type           = 'linear',
            fp16                        = FP16,
            predict_with_generate       = True,   # pakai beam search saat eval
            generation_max_length       = MAX_TARGET_LENGTH,
            evaluation_strategy         = 'epoch',
            save_strategy               = 'epoch',
            logging_strategy            = 'epoch',
            load_best_model_at_end      = True,   # load model terbaik di akhir
            metric_for_best_model       = 'eval_loss',
            greater_is_better           = False,  # semakin kecil loss semakin baik
            save_total_limit            = 2,      # simpan max 2 checkpoint (~4GB)
            report_to                   = 'none', # nonaktifkan wandb/tensorboard
            dataloader_num_workers      = 4,
            seed                        = SEED,
        )

        trainer = Seq2SeqTrainer(
            model           = model,
            args            = training_args,
            train_dataset   = train_dataset,
            eval_dataset    = val_dataset,
            compute_metrics = compute_metrics,
            callbacks       = [
                EarlyStoppingCallback(early_stopping_patience=PATIENCE),
                epoch_cb,
            ],
        )

        print(f'Batch size : {batch_size}  |  FP16 : {FP16}')
        trainer.train(resume_from_checkpoint=resume_checkpoint)
        break   # training selesai tanpa error

    except RuntimeError as e:
        if 'out of memory' in str(e).lower() and attempt == 0:
            # OOM: kurangi batch size dan coba ulang
            batch_size = 8
            torch.cuda.empty_cache()
            print(f'\n[OOM] CUDA kehabisan memori!')
            print(f'[OOM] Mengurangi batch size: {BATCH_SIZE} -> {batch_size}')
            print(f'[OOM] Mencoba ulang...\n')
        else:
            raise   # error lain — lempar ke atas

total_train_time = time.time() - start_time
print('=' * 60)
print(f'Training selesai dalam {total_train_time/60:.1f} menit')
print(f'Batch size yang digunakan : {batch_size}')
if train_dataset.n_corrupt > 0:
    print(f'Gambar corrupt (skipped)  : {train_dataset.n_corrupt}')

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 6 — SIMPAN MODEL & TRAINING HISTORY
# ════════════════════════════════════════════════════════════════

# ── SIMPAN MODEL & PROCESSOR ─────────────────────────────────────
print('Menyimpan model terbaik...')
trainer.save_model(PATH_OUTPUT)
processor.save_pretrained(PATH_OUTPUT)
print(f'Model tersimpan di: {PATH_OUTPUT}')

# ── EKSTRAK METRICS PER EPOCH DARI LOG HISTORY ───────────────────
log_history = trainer.state.log_history

train_losses, val_losses, cer_scores, epochs = [], [], [], []

for entry in log_history:
    if 'eval_loss' in entry:
        epochs.append(int(entry.get('epoch', 0)))
        val_losses.append(round(entry.get('eval_loss', 0), 4))
        cer_scores.append(round(entry.get('eval_cer',  0), 4))

for entry in log_history:
    if 'loss' in entry and 'eval_loss' not in entry:
        train_losses.append(round(entry.get('loss', 0), 4))

best_idx   = val_losses.index(min(val_losses)) if val_losses else 0
best_epoch = epochs[best_idx]   if epochs     else 0
best_cer   = cer_scores[best_idx] if cer_scores else 0

# ── SIMPAN training_history.json ─────────────────────────────────
training_config = {
    'model_base'              : MODEL_NAME,
    'batch_size_used'         : batch_size,
    'batch_size_configured'   : BATCH_SIZE,
    'learning_rate'           : LEARNING_RATE,
    'max_epochs'              : MAX_EPOCHS,
    'actual_epochs'           : len(epochs),
    'warmup_steps'            : WARMUP_STEPS,
    'weight_decay'            : WEIGHT_DECAY,
    'early_stopping_patience' : PATIENCE,
    'fp16'                    : FP16,
    'seed'                    : SEED,
    'max_target_length'       : MAX_TARGET_LENGTH,
    'train_samples'           : len(data_train),
    'val_samples'             : len(data_val),
}

history_json = {
    'training_config' : training_config,
    'epochs'          : epochs,
    'train_loss'      : train_losses,
    'val_loss'        : val_losses,
    'cer'             : cer_scores,
    'best_epoch'      : best_epoch,
    'best_val_loss'   : min(val_losses) if val_losses else None,
    'best_cer'        : best_cer,
    'best_cer_pct'    : round(best_cer * 100, 2),
    'total_time_min'  : round(total_train_time / 60, 1),
    'finished_at'     : datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
}

history_path = os.path.join(PATH_OUTPUT, 'training_history.json')
with open(history_path, 'w', encoding='utf-8') as f:
    json.dump(history_json, f, indent=2, ensure_ascii=False)
print(f'History tersimpan di: {history_path}')

# ── RINGKASAN AKHIR ───────────────────────────────────────────────
print()
print('=' * 55)
print('  RINGKASAN HASIL TRAINING')
print('=' * 55)
print(f'  Total epoch dijalankan : {len(epochs)}')
print(f'  Epoch terbaik          : {best_epoch}')
print(f'  Best val loss          : {min(val_losses):.4f}' if val_losses else '')
print(f'  Best CER               : {best_cer:.4f}  ({best_cer*100:.2f}%)')
print(f'  Total waktu training   : {total_train_time/60:.1f} menit')
print(f'  Model tersimpan di     : {PATH_OUTPUT}')
print('=' * 55)

# ── GRAFIK LOSS & CER ────────────────────────────────────────────
if epochs:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    fig.suptitle('TrOCR Fine-Tuning Results', fontsize=13, fontweight='bold')

    # Plot Loss
    ax = axes[0]
    if train_losses:
        x_train = list(range(1, len(train_losses) + 1))
        ax.plot(x_train, train_losses, 'b-o', label='Train Loss', markersize=4)
    ax.plot(epochs, val_losses, 'r-o', label='Val Loss', markersize=4)
    if best_epoch:
        ax.axvline(x=best_epoch, color='green', linestyle='--',
                   alpha=0.7, label=f'Best epoch {best_epoch}')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.set_title('Training & Validation Loss')
    ax.legend()
    ax.grid(True, alpha=0.3)

    # Plot CER
    ax = axes[1]
    ax.plot(epochs, [c * 100 for c in cer_scores], 'g-o', markersize=4, label='CER (%)')
    if best_epoch:
        ax.axvline(x=best_epoch, color='green', linestyle='--',
                   alpha=0.7, label=f'Best epoch {best_epoch}')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('CER (%)')
    ax.set_title('Character Error Rate (Validation)')
    ax.legend()
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    chart_path = os.path.join(PATH_OUTPUT, 'training_chart.png')
    plt.savefig(chart_path, dpi=120, bbox_inches='tight')
    plt.show()
    print(f'Grafik disimpan di: {chart_path}')
else:
    print('Tidak ada data epoch untuk divisualisasikan.')

print('\nSelesai.')

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 7 — UPLOAD MODEL KE GOOGLE DRIVE (via rclone)
# ════════════════════════════════════════════════════════════════
# Prasyarat: rclone sudah dikonfigurasi (lihat instruksi di Cell 0)
# Jalankan cell ini setelah Cell 6 selesai

import subprocess
import os

# ── KONFIGURASI ──────────────────────────────────────────────────
RCLONE_REMOTE   = 'gdrive'                  # nama remote di rclone config
GDRIVE_DEST_DIR = 'TrOCR-IDP-Lintasarta'   # folder tujuan di Google Drive
LOCAL_MODEL_DIR = PATH_OUTPUT               # 'model_finetuned/'

RCLONE_DEST = f'{RCLONE_REMOTE}:{GDRIVE_DEST_DIR}'

# ── CEK RCLONE TERSEDIA ───────────────────────────────────────────
def check_rclone():
    try:
        result = subprocess.run(['rclone', 'version'], capture_output=True, text=True)
        print(result.stdout.split('\n')[0])
        return True
    except FileNotFoundError:
        print('ERROR: rclone tidak ditemukan.')
        print('Jalankan setup rclone terlebih dahulu (lihat instruksi di Cell 0).')
        return False

# ── CEK REMOTE DAPAT DIAKSES ─────────────────────────────────────
def check_remote():
    result = subprocess.run(
        ['rclone', 'lsd', f'{RCLONE_REMOTE}:'],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print(f'ERROR: Remote "{RCLONE_REMOTE}" tidak dapat diakses.')
        print(result.stderr)
        return False
    return True

# ── UPLOAD ───────────────────────────────────────────────────────
def upload_to_gdrive():
    if not os.path.isdir(LOCAL_MODEL_DIR):
        print(f'ERROR: Folder model tidak ditemukan: {LOCAL_MODEL_DIR}')
        print('Pastikan Cell 6 sudah dijalankan terlebih dahulu.')
        return

    print(f'Mengupload {LOCAL_MODEL_DIR} → {RCLONE_DEST}/')
    print('(Proses ini bisa memakan waktu beberapa menit...)\n')

    result = subprocess.run(
        ['rclone', 'copy', LOCAL_MODEL_DIR, RCLONE_DEST,
         '--progress', '--transfers=4', '--checksum'],
        text=True
    )

    if result.returncode == 0:
        print(f'\nUpload berhasil!')
        print(f'Model tersedia di Google Drive: {RCLONE_DEST}/')
        print('\nFile yang diupload:')
        list_result = subprocess.run(
            ['rclone', 'ls', RCLONE_DEST],
            capture_output=True, text=True
        )
        print(list_result.stdout)
    else:
        print(f'\nUpload gagal (exit code {result.returncode})')
        print(result.stderr)

# ── JALANKAN ─────────────────────────────────────────────────────
print('=== Upload Model ke Google Drive ===')
print(f'Remote  : {RCLONE_REMOTE}')
print(f'Sumber  : {LOCAL_MODEL_DIR}')
print(f'Tujuan  : {RCLONE_DEST}/')
print()

if check_rclone() and check_remote():
    upload_to_gdrive()